### Visualize optimization

In [ ]:
# Load the vars
import pickle
from utils.filemanager import get_latest_temp_subfolder

# data_file = "results/sabater2d/11_07_26_Sabater2d_ndoe_-1_n_EI_-1/ro_data.pkl"
data_file = r'latest'
if data_file == "latest":
    latest_temp_folder = get_latest_temp_subfolder()
    data_file = latest_temp_folder + "/ro_data.pkl"
    print(f"loading: {data_file}")

loaded_objects = []
with open(f'{data_file}', 'rb') as f:
    try:
        while True:
            loaded_objects.append(pickle.load(f))
    except EOFError:
        pass

ro_data, model, gp_model, args = loaded_objects
ref_x, ref_q = model.load_opt(measure = args.measure, tau = args.tau, alpha = args.alpha)
ref_plot = model.load_domain_plot(measure = args.measure, tau = args.tau, alpha = args.alpha)
ref_domain, ref_q_vec = ref_plot["x"], ref_plot["q"]
oat_plot = model.load_oat_plot(measure = args.measure, tau = args.tau, alpha = args.alpha)

### DOE Phase

In [ ]:
import matplotlib.pyplot as plt
import numpy as np 
from utils.plotting import contour_plot

plt_doe = ro_data.doe["plot_data"][0]
domain = plt_doe["x_domain"]

doe_xtilde = ro_data.doe["xtilde"]
J_vec = plt_doe["J_mean_vec"]
J_std_vec = plt_doe["J_std_vec"]

fig, ax = plt.subplots(1,3, figsize = [9,3])
ax[0] = contour_plot(ref_domain, ref_q_vec, ax = ax[0])
ax[0].scatter(doe_xtilde[:,0], doe_xtilde[:,1])
ax[0].set_title("Reference")
ax[1] = contour_plot(domain, J_vec, ax = ax[1])
ax[1].set_title("Quant Mean")
ax[2] = contour_plot(domain, J_std_vec, ax = ax[2])
ax[2].set_title("Quant Std")

if ro_data.doe_infill["n"] > 0:
    plt_infill = ro_data.doe_infill["plot_data"][0]
    quant_mean_vec_infill = plt_infill["J_mean_vec"]
    quant_std_vec_infill = plt_infill["J_std_vec"]
    fig, ax = plt.subplots(1,3, figsize = [9,3])
    ax[0] = contour_plot(ref_domain, ref_q_vec, ax = ax[0])
    ax[0].scatter(doe_xtilde[:,0], doe_xtilde[:,1])
    ax[0].set_title("Reference")
    ax[1] = contour_plot(domain, quant_mean_vec_infill, ax = ax[1])
    ax[1].set_title("Quant Mean")
    ax[2] = contour_plot(domain, quant_std_vec_infill, ax = ax[2])
    ax[2].set_title("Quant Std")
    ax[2].scatter(doe_xtilde[:,0], doe_xtilde[:,1])

plt.show()

### EI Phase

In [ ]:
import matplotlib.pyplot as plt
import numpy as np 
from utils.plotting import create_oat_plot

ccycle = plt.rcParams['axes.prop_cycle'].by_key()['color']

n_EI = ro_data.EI_infill["n"]
cand_x_EI = np.array(ro_data.EI_infill["cand_x"])
EI_xtilde = ro_data.doe["xtilde"]
EIvals = []

print(n_EI)

for i in range(n_EI):
    fig, ax = plt.subplots(1, 4, figsize = [16,3])

    data = ro_data.EI_infill["plot_data"][i]
    domain = data["x_domain"]
    J_vec = data["J_mean_vec"]
    J_std_vec = data["J_std_vec"]
    JEI = data["JEI"]
    x_star = data["x_star"]
    J_star = data["J_star"]
    sweep_plots = data["sweep_plots"]
    EIvals.append(data["EI_star"])

    ax[0] = contour_plot(ref_domain, ref_q_vec, levels = 30, cbar = True, ax = ax[0])
    ax[0].scatter(doe_xtilde[:,0], doe_xtilde[:,1], s = 5, color = "grey")
    ax[0].scatter(cand_x_EI[:i,0], cand_x_EI[:i,1],  marker = "+", color = "grey")
    ax[0].set_title("Full model quantile")
    
    ax[1] = contour_plot(domain, np.mean(J_vec.numpy(), axis = 1), levels = 30, cbar = True, ax = ax[1])
    ax[1].scatter(x_star[0], x_star[1], marker = "*", color = "r", label = "Best cand")
    ax[1].set_title("GP model Mean quantile")
    ax[1].legend()
    ax[1].axis("off")

    ax[2] = contour_plot(domain, np.mean(J_std_vec.numpy(),axis = 1), levels = 30, cbar = True, ax = ax[2])
    ax[2].scatter(cand_x_EI[:i,0], cand_x_EI[:i,1], marker = "+", color = "grey")
    ax[2].set_title("GP model Std quantile")
    ax[2].axis("off")

    ax[3] = contour_plot(domain, np.mean(JEI.numpy(), axis = 1), levels = 30, cbar = True, ax = ax[3])
    ax[3].set_title("EI contour plot")
    ax[3].scatter(cand_x_EI[i,0], cand_x_EI[i,1], s = 30, marker = "+", color = "g")
    ax[3].axis("off")

### EM Infill

In [ ]:
n_EM = ro_data.EM_infill["n"]
cand_x_EM = np.array(ro_data.EM_infill["cand_x"])

for i in range(n_EM):
    fig, ax = plt.subplots(1, 2, figsize = [5,3])

    data = ro_data.EM_infill["plot_data"][i]
    domain = data["x_domain"]
    J_vec = data["J_vec"]
    x_c = data["x_c"]
    J_c = data["J_c"]
    x_star = data["x_star"]
    J_star = data["J_star"]
    sweep_plots = data["sweep_plots"]

    x_star_post = ro_data.EM_infill["best_x"][i]
    q_star_post = ro_data.EM_infill["best_J"][i]

    ax[0] = contour_plot(ref_domain, ref_q_vec, levels = 30, cbar = True, ax = ax[0])
    ax[0].scatter(doe_xtilde[:,0], doe_xtilde[:,1], s = 5, color = "grey")
    ax[0].scatter(cand_x_EI[:,0], cand_x_EI[:,1],  marker = "+", color = "grey")
    ax[0].scatter(cand_x_EM[:i,0], cand_x_EM[:i,1],  marker = "+", color = "g")
    ax[0].set_title("Full model quantile")
    
    ax[1] = contour_plot(domain, np.mean(J_vec.numpy(), axis = 1), levels = 30, cbar = True, ax = ax[1])
    ax[1].scatter(x_star[0], x_star[1], marker = "*", color = "r", label = "Best cand")
    ax[1].scatter(x_c[0], x_c[1], marker = "+", color = "r", label = "Proposal")
    ax[1].set_title("GP model Mean quantile")
    ax[1].legend()
    ax[1].axis("off")

    print(f"Iteration {i+1}: Pre: ")
    print(f"x_c    = {x_c.numpy()}")
    print(f"x_star = {x_star.numpy()} with q_star = {J_star}")
    print(f"Iteration {i+1}: Post: ")
    print(f"x_star = {x_star_post.numpy()} with q_star = {q_star_post} \n")

### TR Iteration

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import matplotlib.patches as patches

ccycle = plt.rcParams['axes.prop_cycle'].by_key()['color']

n_TR = ro_data.TR_infill["n"]
cand_x_TR = np.array(ro_data.TR_infill["cand_x"])

for i in range(n_TR):
    fig, ax = plt.subplots(1, 2, figsize = [12,3])

    data = ro_data.TR_infill["plot_data"][i]
    domain = data["x_domain"]
    J = data["J"]
    x_star = data["x_star"]
    x_c = data["x_c"]
    trust_region = data["trust_reg"]

    ax[0] = contour_plot(ref_domain, ref_q_vec, levels = 30, cbar = True, ax = ax[0])
    ax[0].scatter(doe_xtilde[:,0], doe_xtilde[:,1], s = 5, color = "grey")
    ax[0].scatter(cand_x_EI[:,0], cand_x_EI[:,1],  marker = "+", color = "grey")
    ax[0].scatter(cand_x_TR[:i,0], cand_x_TR[:i,1], marker = "+", color = "g")
    ax[0].set_title("Full model quantile")
    
    # Create a rectangle
    rect = patches.Rectangle((trust_region[0,0], trust_region[0,1]),  # bottom-left corner
                            trust_region[1,0] - trust_region[0,0],  # width
                            trust_region[1,1] - trust_region[0,1],  # height
                            linewidth=2, edgecolor='r', facecolor='none', alpha=0.5)

    ax[1] = contour_plot(domain, J.numpy(), levels = 30, cbar = True, ax = ax[1])
    ax[1].scatter(x_star[0], x_star[1], marker = "*", color = "r", label = "Best cand")
    ax[1].scatter(x_c[0], x_c[1], marker = "+", color = "r", label = "Best cand")
    ax[1].add_patch(rect)
    ax[1].set_title("GP model quantile")
    ax[1].legend()
    ax[1].axis("off")


### Candidate Evolution

In [ ]:
from utils.plotting import plot_cand_evolution

plot_cand_evolution(ro_data, model, args)

### Best cand evolution

In [ ]:
from utils.plotting import plot_best_evolution

plot_best_evolution(ro_data, model, args)

### Convergence plots

In [ ]:
from utils.plotting import plot_convergence

plot_convergence(ro_data, model, args)